<a href="https://colab.research.google.com/github/BOBBYKIRAN/ai-mentor-portfolio/blob/main/Day9_StudyAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [4]:
!
!pip install -U ddgs
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.7 MB/s eta 0:00:00
February 25, 2026 - TCS will conduct the hiring round using the TCS iON platform rather than the TCS Community platform, which TCS has utilized in the past. Yes, students from 2024 can apply in this drive. January 20, 2026 - Tata Consultancy Services (TCS), India’s most trusted IT services company, has launched its TCS Hiring 2026 drive, creating a massive wave of job opportunities for freshers an


In [5]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_826/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [6]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': '352128db-4e9b-42c7-89b4-4fafba0468bf', 'type': 'tool_call'}]

[2] ToolMessage
    Content: 1 week ago - TCS NQT 2026 Recruitment is open for 2024–2026 batches. Check eligibility, salary up to ₹12.26 LPA, exam pattern, important dates, and how to apply online. 1 week ago - For 2026, TCS has retained the 60% aggregate threshold introduced in 2022 but tightened its standing arrears policy, c

[3] AIMessage
    Content: [{'type': 'text', 'text': 'I could not find a specific hiring quota for TCS for 2026 in my search results. However, TCS has announced the NQT (National Qualifier Test) hiring drive for the 2026 batch across India, targeting candidates from the 2024, 2025, and 2026 batches for Prime and Digital roles


In [7]:
# Pass a question that should fail the tool
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

# Watch how the agent recovers
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    

[2] ToolMessage
    It will give you the status of whether the domain name that you are looking for is available or not. Doing this for multiple domain names over time can help you build a list of available domain names. Use our WHOIS lookup tool to check domain name availability or to discover the contact information 

[3] AIMessage
    [{'type': 'text', 'text': 'The domain "this-domain-does-not-exist-12345.example.com" likely does not exist. The search results provided information on how to check if a domain name is available, WHOIS lookup tools, and how to find alternative domain names. It also mentioned that if you query a domai
